# Video Emotion Analysis\n\nThis notebook trains a video emotion model from `vediodataset` and saves it as `mlmodel/video_emotion_model.pkl`.\n\nSupported dataset layouts:\n\n1. Folder labels: `vedio dataset/happy/sample1.mp4`, `vedio dataset/sad/sample2.mp4`, etc.\n2. Metadata CSV: `vedio dataset/metadata.csv` with `video_path` and `emotion` columns.\n\nThe current local environment has OpenCV for face/frame reaction features. Voice-tone features are read from `.wav` sidecar files with the same basename as each video when available, for example `sample1.mp4` plus `sample1.wav`.

In [ ]:
from pathlib import Path\nimport pickle\n\nimport pandas as pd\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import accuracy_score, classification_report\nfrom sklearn.model_selection import train_test_split\n\nfrom mlmodel.video_features import EMOTIONS, VIDEO_EXTENSIONS, extract_video_features, video_feature_names

In [ ]:
dataset_candidates = [Path('vedio dataset'), Path('vediodataset'), Path('video dataset'), Path('videodataset')]\ndataset_dir = next((path for path in dataset_candidates if path.exists()), dataset_candidates[0])\nmodel_path = Path('mlmodel/video_emotion_model.pkl')\n\ndef load_folder_records(dataset_dir):\n    records = []\n    for emotion_dir in dataset_dir.iterdir():\n        if not emotion_dir.is_dir():\n            continue\n        label = emotion_dir.name.strip().lower()\n        if label not in EMOTIONS:\n            continue\n        for video_path in emotion_dir.rglob('*'):\n            if video_path.is_file() and video_path.suffix.lower() in VIDEO_EXTENSIONS:\n                records.append((video_path, label))\n    if records:\n        return records\n    for video_path in dataset_dir.rglob('*'):\n        if not video_path.is_file() or video_path.suffix.lower() not in VIDEO_EXTENSIONS:\n            continue\n        normalized = video_path.stem.lower()\n        for emotion in EMOTIONS:\n            if f'_{emotion}' in normalized or f'-{emotion}' in normalized or f' {emotion}' in normalized:\n                records.append((video_path, emotion))\n                break\n    return records\n\ndef load_metadata_records(metadata_path):\n    df = pd.read_csv(metadata_path)\n    path_column = next((column for column in ['video_path', 'path', 'file', 'filename'] if column in df.columns), None)\n    label_column = next((column for column in ['emotion', 'label', 'class'] if column in df.columns), None)\n    if path_column is None or label_column is None:\n        raise ValueError('Metadata CSV must contain video_path and emotion columns.')\n    records = []\n    for _, row in df.iterrows():\n        label = str(row[label_column]).strip().lower()\n        video_path = Path(str(row[path_column]).strip())\n        if not video_path.is_absolute():\n            video_path = dataset_dir / video_path\n        if label in EMOTIONS and video_path.exists() and video_path.suffix.lower() in VIDEO_EXTENSIONS:\n            records.append((video_path, label))\n    return records\n\nmetadata_path = dataset_dir / 'metadata.csv'\nrecords = load_metadata_records(metadata_path) if metadata_path.exists() else load_folder_records(dataset_dir)\nlen(records), records[:3]

In [ ]:
features = []\nlabels = []\n\nfor index, (video_path, label) in enumerate(records, start=1):\n    print(f'[{index}/{len(records)}] {video_path}')\n    features.append(extract_video_features(video_path))\n    labels.append(label)\n\nx = pd.DataFrame(features, columns=video_feature_names())\ny = pd.Series(labels)\nx.head(), y.value_counts()

In [ ]:
stratify = y if y.value_counts().min() >= 2 else None\nx_train, x_test, y_train, y_test = train_test_split(\n    x, y, test_size=0.2, random_state=42, stratify=stratify\n)\n\nmodel = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')\nmodel.fit(x_train, y_train)

In [ ]:
predictions = model.predict(x_test)\nprint('Accuracy:', accuracy_score(y_test, predictions))\nprint(classification_report(y_test, predictions))

In [ ]:
with model_path.open('wb') as model_file:\n    pickle.dump(model, model_file)\n\nmodel_path